# ETL Pipeline — Tarifas CENS
**Plataforma de Inteligencia Competitiva VATIA — MVP**

Flujo: Descarga automática del **PDF de Tarifas** desde la web de CENS → Extracción de la tabla de componentes CU con **pdfplumber** → Exportación de CSV listo para Power BI.

**Fuente de datos:** Columna _Periodo_ de la tabla en `cens.com.co` → archivos `Tarifas_CENS_AAAAMM_.pdf`

**Salida esperada:** `tarifas_cens_limpio.csv` con columnas `G;T;D;Cv;PR;R;CU` por nivel de tensión, separador `;` y decimales con coma (formato latino).


## 0. Instalación de Dependencias
Ejecutar solo si es necesario (Colab o entorno nuevo).

In [36]:
import subprocess, sys
# Instalar dependencias faltantes de forma silenciosa
# NOTA: easyocr descarga ~200 MB de modelos en el primer uso (solo una vez)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "requests", "beautifulsoup4", "pymupdf", "easyocr"])
print("✔ Dependencias instaladas.")


✔ Dependencias instaladas.


## 1. Importaciones y Configuración Global

In [48]:
import re
import os
import io
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
import pdfplumber

# ─── CONFIGURACIÓN ───────────────────────────────────────────────────────────
URL_TARIFAS     = "https://www.cens.com.co/clientes-y-usuarios/tarifas-de-energia"
COMERCIALIZADOR = "CENS"
MERCADO         = "CENS"
DUENO_RED       = "100% OPERADOR"
OUTPUT_CSV      = "tarifas_cens_limpio.csv"

# ── Mapeo de componentes CU: patrón regex (fila del PDF) → nombre columna destino
# Fuente: Tabla "COMPONENTES DEL COSTO UNITARIO- CU en $/kWh" del PDF de Tarifas
# Filas esperadas: G | T | DtUN | Cv | PR | R | CUv
MAPEO_COMPONENTES = [
    (r"^G$",           "G"),    # Generación           ($/kWh)
    (r"^T$",           "T"),    # Transmisión          ($/kWh)
    (r"DtUN|DtN|^D$",  "D"),    # Distribución NT      ($/kWh)
    (r"^Cv$",          "Cv"),   # Comercialización var ($/kWh)
    (r"^PR$",          "PR"),   # Pérdidas recono.     ($/kWh)
    (r"^R$",           "R"),    # Restricciones        ($/kWh)
    (r"CUv|^CU$",      "CU"),   # Costo Unitario total ($/kWh)
]

# ── Mapeo de niveles: patrón regex (columna del PDF) → valor Nivel_Tension
# Solo se extraen los 4 niveles principales (no Compartido/Particular)
# Columnas del PDF: 1-2,CENS | 1-2 Compartido | 1-2,Particular | Nivel 2 | Nivel 3 | Nivel 4
MAPEO_NIVELES_PDF = [
    (r"1-2.*CENS",  "1"),   # "1-2,CENS" → Nivel de Tensión 1 (operado por CENS)
    (r"Nivel\s*2",  "2"),   # Nivel 2
    (r"Nivel\s*3",  "3"),   # Nivel 3
    (r"Nivel\s*4",  "4"),   # Nivel 4
]
# Nota: "1-2 Compartido" y "1-2,Particular" no se extraen (no están en el prototipo de datos)

print("✔ Configuración cargada.")
print(f"  Componentes : {[c for _, c in MAPEO_COMPONENTES]}")
print(f"  Niveles PDF : {[n for _, n in MAPEO_NIVELES_PDF]}")


✔ Configuración cargada.
  Componentes : ['G', 'T', 'D', 'Cv', 'PR', 'R', 'CU']
  Niveles PDF : ['1', '2', '3', '4']


## 2. Extracción — Descarga del PDF de Tarifas


In [49]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}


def obtener_enlaces_pdf(url: str) -> list[tuple[str, str]]:
    """
    Raspa la página de CENS y devuelve TODOS los archivos PDF de Tarifas disponibles.

    Estrategia:
      1. Localiza la <table> con encabezado 'Periodo'.
      2. Recoge todos los hrefs .pdf de la columna 'Periodo' (columna 0).
      3. Devuelve lista de (nombre_archivo, url_absoluta) ordenada por ciclo.

    Returns:
        Lista de tuplas (nombre_archivo, url_descarga).
        Ej: [('Tarifas_CENS_202601_.pdf', 'https://...'), ...]
    """
    print(f"[SCRAPING] Accediendo a: {url}")
    respuesta = requests.get(url, headers=HEADERS, timeout=30)
    respuesta.raise_for_status()

    soup = BeautifulSoup(respuesta.text, "html.parser")
    enlaces: list[tuple[str, str]] = []

    for tabla in soup.find_all("table"):
        encabezados = [th.get_text(strip=True) for th in tabla.find_all("th")]

        # La tabla de tarifas tiene "Periodo" como primer encabezado
        if not encabezados or not re.search(r"periodo|per[ií]odo", encabezados[0], re.I):
            continue

        print(f"  ✔ Tabla encontrada con encabezados: {encabezados}")

        for fila in tabla.find_all("tr"):
            celdas = fila.find_all("td")
            if not celdas:
                continue
            # Columna 0 = "Periodo" → contiene el enlace al PDF de Tarifas
            tag_a = celdas[0].find("a", href=True)
            if tag_a:
                href = tag_a["href"]
                if ".pdf" in href.lower():
                    # Construir URL absoluta y limpiar el "?" al final
                    href_limpio = href.split("?")[0]
                    if href_limpio.startswith("http"):
                        url_descarga = href_limpio
                    else:
                        url_descarga = f"https://www.cens.com.co{href_limpio}"
                    nombre = url_descarga.split("/")[-1]
                    ciclo_match = re.search(r"\d{6}", nombre)
                    ciclo_label = ciclo_match.group() if ciclo_match else nombre
                    print(f"  → PDF Tarifas: {nombre}  (ciclo {ciclo_label})")
                    enlaces.append((nombre, url_descarga))
        break  # Solo procesar la primera tabla con "Periodo"

    if not enlaces:
        # Fallback: buscar href que contenga "Tarifas" y termine en .pdf
        print("  ℹ Tabla 'Periodo' no detectada. Buscando por patrón en href...")
        for tag_a in soup.find_all("a", href=True):
            href = tag_a["href"]
            if re.search(r"Tarifas.*\.pdf", href, re.I):
                href_limpio = href.split("?")[0]
                url_descarga = href_limpio if href_limpio.startswith("http") \
                    else f"https://www.cens.com.co{href_limpio}"
                nombre = url_descarga.split("/")[-1]
                enlaces.append((nombre, url_descarga))
                print(f"  → PDF fallback: {nombre}")

    if not enlaces:
        raise ValueError(
            "No se encontró ningún enlace PDF en la página. "
            "Verifique la URL o actualice el patrón de búsqueda."
        )

    # Ordenar por ciclo (AAAAMM) de menor a mayor
    enlaces.sort(
        key=lambda t: re.search(r"\d{6}", t[0]).group()
        if re.search(r"\d{6}", t[0]) else t[0]
    )
    print(f"\n[SCRAPING] Total archivos PDF disponibles: {len(enlaces)}")
    return enlaces


def descargar_archivo(url_descarga: str, nombre_archivo: str) -> bytes:
    """
    Descarga un archivo desde una URL y devuelve su contenido en bytes.
    """
    print(f"  ↓ Descargando {nombre_archivo} ...", end=" ")
    respuesta = requests.get(url_descarga, headers=HEADERS, timeout=60)
    respuesta.raise_for_status()
    print(f"{len(respuesta.content) / 1024:.1f} KB")
    return respuesta.content


print("✔ Funciones de extracción definidas.")


✔ Funciones de extracción definidas.


## 3. Transformación — Extracción de Componentes CU del PDF


In [47]:
import fitz       # PyMuPDF — renderiza páginas PDF como imágenes
import easyocr    # OCR puro Python — extrae texto de imágenes
import numpy as np

# ── Inicializar lector OCR una sola vez (evita re-cargar modelos)
# NOTA: En el primer uso descarga los modelos (~200 MB) a ~/.EasyOCR/
_ocr_reader: easyocr.Reader | None = None

def _get_ocr_reader() -> easyocr.Reader:
    global _ocr_reader
    if _ocr_reader is None:
        print("  Inicializando OCR (primera vez: descarga modelos ~200 MB)...")
        _ocr_reader = easyocr.Reader(['en'], verbose=False)
        print("  OCR listo.")
    return _ocr_reader


def _extraer_ciclo_y_fecha(nombre_archivo: str) -> tuple[str, str]:
    match = re.search(r"(\d{6})", nombre_archivo)
    if not match:
        raise ValueError(
            f"No se pudo extraer el ciclo del nombre: '{nombre_archivo}'. "
            "Se esperaba AAAAMM (ej. 202601)."
        )
    ciclo = match.group(1)
    fecha = datetime(int(ciclo[:4]), int(ciclo[4:]), 1).strftime("%Y-%m-%d")
    return ciclo, fecha


def _parse_numero(valor) -> float | None:
    """Convierte texto extraído por OCR a float, tolerando variantes de formato."""
    s = str(valor or "").strip()
    if not s or s.lower() in ("none", "nan", "-", "n/a", ""):
        return None
    # Limpiar artefactos OCR comunes: espacios, saltos de línea, letras sueltas
    s = re.sub(r"[^\d.,]", "", s)
    if not s:
        return None
    # "1.234,56" → separador miles=punto, decimal=coma
    if "." in s and "," in s:
        s = s.replace(".", "").replace(",", ".")
    elif "," in s:
        s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


def _ocr_pagina(pdf_bytes: bytes, pagina_idx: int, escala: float = 2.5) -> list:
    """
    Renderiza una página del PDF y aplica OCR.

    Returns:
        Lista de (y_centro, x_centro, texto) para cada elemento detectado.
    """
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    page = doc[pagina_idx]
    pix  = page.get_pixmap(matrix=fitz.Matrix(escala, escala))
    # Convertir a numpy array RGB
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)
    if pix.n == 4:            # RGBA → RGB
        img = img[:, :, :3]

    reader   = _get_ocr_reader()
    results  = reader.readtext(img, detail=1, paragraph=False)

    items = []
    for bbox, texto, conf in results:
        if conf < 0.4:        # Descartar detecciones de baja confianza
            continue
        xc = (bbox[0][0] + bbox[2][0]) / 2
        yc = (bbox[0][1] + bbox[2][1]) / 2
        items.append((yc, xc, texto.strip()))

    return items


def _agrupar_en_filas(items: list, tolerancia_y: float = 18.0) -> list[list[str]]:
    """
    Agrupa items OCR en filas (misma Y ± tolerancia) y las ordena por X.
    Returns: lista de listas de strings, una por fila, ordenada de arriba a abajo.
    """
    if not items:
        return []

    items_ord = sorted(items, key=lambda x: x[0])
    filas: list[list] = []
    fila_actual = [items_ord[0]]
    y_ref = items_ord[0][0]

    for item in items_ord[1:]:
        if abs(item[0] - y_ref) <= tolerancia_y:
            fila_actual.append(item)
        else:
            fila_actual.sort(key=lambda x: x[1])
            filas.append([t for _, _, t in fila_actual])
            fila_actual = [item]
            y_ref = item[0]

    if fila_actual:
        fila_actual.sort(key=lambda x: x[1])
        filas.append([t for _, _, t in fila_actual])

    return filas


def _parsear_filas_ocr(
    filas: list[list[str]],
    mapeo_componentes: list[tuple[str, str]],
    mapeo_niveles: list[tuple[str, str]],
) -> dict[str, dict[str, float | None]]:
    """
    Reconstruye el diccionario {nivel: {componente: valor}} a partir de
    las filas OCR agrupadas.

    Estrategia:
      1. Busca la fila de encabezado de columnas (contiene "CENS" o "Nivel N")
      2. Las filas siguientes son las filas de componentes (G, T, DtUN, …)
      3. Mapea posición de columna → nivel de tensión
      4. Mapea primera celda de fila → nombre de componente
    """
    # 1. Encontrar fila con encabezados de nivel
    header_idx = None
    for i, fila in enumerate(filas):
        texto = " ".join(fila)
        if re.search(r"CENS|Nivel\s*\d|Compartido|Particular", texto, re.I):
            header_idx = i
            break

    if header_idx is None:
        raise ValueError(
            "No se encontró la fila de encabezados de nivel en el texto OCR. "
            f"Filas disponibles: {[' | '.join(f) for f in filas[:8]]}"
        )

    fila_header = filas[header_idx]
    print(f"    Encabezados de nivel (fila {header_idx}): {fila_header}")

    # 2. Mapear posición de columna → nombre de nivel
    # La columna 0 es la etiqueta del componente; los niveles empiezan en col 1
    col_to_nivel: dict[int, str] = {}
    for col_idx, celda in enumerate(fila_header):
        if col_idx == 0:
            continue
        for patron, nivel_name in mapeo_niveles:
            if re.search(patron, celda, re.I):
                col_to_nivel[col_idx] = nivel_name
                break

    if not col_to_nivel:
        # Fallback: si la fila 0 del header tiene solo un elemento (título merged),
        # el header real puede estar en la siguiente fila
        if header_idx + 1 < len(filas):
            fila_header = filas[header_idx + 1]
            for col_idx, celda in enumerate(fila_header):
                if col_idx == 0:
                    continue
                for patron, nivel_name in mapeo_niveles:
                    if re.search(patron, celda, re.I):
                        col_to_nivel[col_idx] = nivel_name
                        break
            if col_to_nivel:
                header_idx += 1
                print(f"    Encabezados (fila ajustada {header_idx}): {fila_header}")

    if not col_to_nivel:
        raise ValueError(
            f"No se mapeó ningún nivel de tensión. "
            f"Fila analizada: {fila_header}\n"
            f"Patrones esperados: {[p for p, _ in mapeo_niveles]}"
        )

    print(f"    Niveles mapeados: { {v: k for k, v in col_to_nivel.items()} }")

    # 3. Inicializar estructura y procesar filas de datos
    data: dict[str, dict[str, float | None]] = {n: {} for n in col_to_nivel.values()}

    # Para filas donde el OCR no detectó la etiqueta del componente (ej. G y T,
    # caracteres únicos con baja visibilidad), se asignan en orden secuencial.
    comp_names_ordered = [name for _, name in mapeo_componentes]
    asignados: set[str] = set()
    n_cols_esperadas = len(fila_header)

    for fila in filas[header_idx + 1:]:
        if not fila:
            continue

        # Detectar fila sin etiqueta:
        #   - len = n_cols_esperadas-1 : header completo, solo falta la etiqueta
        #   - len = n_cols_esperadas   : header tambien perdio una col (ej. Particular)
        #   G y T siempre tienen el mismo valor en cada columna (componente nacional).
        etiqueta_faltante = (
            len(fila) in (n_cols_esperadas - 1, n_cols_esperadas)
            and all(_parse_numero(v) is not None for v in fila)
            and len(set(fila)) == 1   # G y T identicos en todas las columnas
        )
        if etiqueta_faltante:
            fila = [""] + fila   # Realinear: insertar etiqueta vacía al frente

        etiqueta = fila[0].strip()

        componente = None
        for patron, comp_name in mapeo_componentes:
            if re.search(patron, etiqueta, re.I):
                componente = comp_name
                break

        # Fallback posicional para filas sin etiqueta
        if componente is None and etiqueta == "":
            for cn in comp_names_ordered:
                if cn not in asignados:
                    componente = cn
                    print(f"    ⚠ Etiqueta no detectada → asignado como '{cn}' "
                          f"(valores: {fila[1:4]}…)")
                    break

        if componente is None:
            continue

        asignados.add(componente)
        for col_idx, nivel in col_to_nivel.items():
            valor_raw = fila[col_idx] if col_idx < len(fila) else None
            data[nivel][componente] = _parse_numero(valor_raw)

    return data


def _encontrar_pagina_tabla(pdf_bytes: bytes) -> int:
    """
    Determina qué página del PDF contiene la tabla de componentes CU.
    Intenta texto primero; si no hay texto en una página, asume que es imagen → OCR.
    Retorna el índice (0-based) de la primera página sin texto (que debe ser la tabla).
    """
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    for i, page in enumerate(doc):
        texto = page.get_text()
        if not texto.strip():
            return i  # Primera página sin texto = tabla como imagen
    return len(doc) - 1  # Fallback: última página


def procesar_pdf(
    pdf_bytes: bytes,
    nombre_archivo: str,
    mapeo_componentes: list[tuple[str, str]] = MAPEO_COMPONENTES,
    mapeo_niveles: list[tuple[str, str]] = MAPEO_NIVELES_PDF,
) -> pd.DataFrame:
    """
    Extrae la tabla de componentes CU (G, T, D, Cv, PR, R, CU) de un PDF de Tarifas CENS.

    Las páginas de tabla están embebidas como imágenes → se usa OCR (easyocr + PyMuPDF).

    Returns:
        DataFrame con una fila por nivel de tensión y columnas G, T, D, Cv, PR, R, CU.
    """
    ciclo, fecha = _extraer_ciclo_y_fecha(nombre_archivo)
    print(f"  Ciclo: {ciclo}  →  Fecha: {fecha}")

    # Detectar página con la tabla CU (primera sin texto extraíble → imagen)
    pagina_idx = _encontrar_pagina_tabla(pdf_bytes)
    print(f"  Tabla CU detectada en página {pagina_idx + 1} (imagen → OCR)")

    # OCR sobre esa página
    items = _ocr_pagina(pdf_bytes, pagina_idx)
    print(f"  OCR: {len(items)} elemento(s) detectados")

    if not items:
        raise RuntimeError(
            f"OCR no detectó ningún texto en la página {pagina_idx + 1}. "
            "Intente con otra escala o verifique el PDF."
        )

    # Agrupar en filas
    filas = _agrupar_en_filas(items)
    print(f"  Filas OCR agrupadas: {len(filas)}")

    # Parsear tabla
    data_por_nivel = _parsear_filas_ocr(filas, mapeo_componentes, mapeo_niveles)

    # Construir DataFrame
    df_filas = []
    for nivel, componentes in data_por_nivel.items():
        fila: dict = {
            "Fecha":           fecha,
            "Ciclo":           ciclo,
            "Comercializador": COMERCIALIZADOR,
            "Mercado":         MERCADO,
            "Nivel_Tension":   nivel,
            "Dueño_Red":       DUENO_RED,
        }
        for _, comp_name in mapeo_componentes:
            fila[comp_name] = componentes.get(comp_name)
        df_filas.append(fila)

    if not df_filas:
        raise RuntimeError(f"No se extrajeron filas de '{nombre_archivo}'.")

    df = pd.DataFrame(df_filas)
    print(f"  → DataFrame: {len(df)} filas × {len(df.columns)} columnas")
    return df


print("✔ Funciones de transformación PDF (OCR) definidas.")


✔ Funciones de transformación PDF (OCR) definidas.


## 4. Carga — Exportación del CSV

In [31]:
def generar_csv(df: pd.DataFrame, ruta_salida: str = OUTPUT_CSV) -> str:
    """
    Exporta el DataFrame a CSV con formato latino:
    - Separador columnas : punto y coma (;)
    - Separador decimal  : coma (,)
    - Codificación       : UTF-8 con BOM (compatible con Excel en español)
    """
    # Redondear todas las columnas numéricas a 4 decimales
    for col in df.select_dtypes(include="number").columns:
        df[col] = df[col].round(4)

    df.to_csv(
        ruta_salida,
        sep=";",
        decimal=",",
        index=False,
        encoding="utf-8-sig"
    )
    ruta_abs = os.path.abspath(ruta_salida)
    print(f"\n✔ CSV exportado: {ruta_abs}")
    return ruta_abs


print("✔ Función generar_csv() definida.")


✔ Función generar_csv() definida.


## 5. Ejecución del Pipeline Completo

In [44]:
# ── Prueba rápida con el PDF local ───────────────────────────────────────────
RUTA_LOCAL_PDF = r"C:\Users\geoff\Desktop\Reto VATIA\Tarifas_CENS_202601_.pdf"

with open(RUTA_LOCAL_PDF, "rb") as f:
    contenido_pdf_local = f.read()

df_prueba = procesar_pdf(contenido_pdf_local, "Tarifas_CENS_202601_.pdf")
print("\n── Resultado ──────────────────────────────────────────────────────────")
print(df_prueba[["Nivel_Tension","G","T","D","Cv","PR","R","CU"]].to_string(index=False))


  Ciclo: 202601  →  Fecha: 2026-01-01
  Tabla CU detectada en página 4 (imagen → OCR)
  Inicializando OCR (primera vez: descarga modelos ~200 MB)...
  OCR listo.


C:\Users\geoff\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  OCR: 153 elemento(s) detectados
  Filas OCR agrupadas: 38
    Encabezados de nivel (fila 6): ['Componentes Cuv', '1-2,CENS', '1-2 Compartido', '1-2,Particular', 'Nivel 2', 'Nivel 3', 'Nivel 4']
    Niveles mapeados: {'1': 1, '2': 4, '3': 5, '4': 6}
    ⚠ Etiqueta no detectada → asignado como 'G' (valores: ['298.9047', '298.9047', '298.9047']…)
    ⚠ Etiqueta no detectada → asignado como 'T' (valores: ['52.9743', '52.9743', '52.9743']…)
  → DataFrame: 4 filas × 13 columnas

── Resultado ──────────────────────────────────────────────────────────
Nivel_Tension        G       T        D       Cv      PR       R       CU
            1 298.9047 52.9743 317.1201 132.3776 67.3480 19.3343 888.0590
            2 298.9047 52.9743 198.4778 120.1964 23.0861 19.3343 712.9736
            3 298.9047 52.9743  88.9777 121.7416 23.5973 19.3343 605.5299
            4 298.9047 52.9743  35.2509  85.2814 13.2552 19.3343 505.0008


In [ ]:
def ejecutar_pipeline(url: str = URL_TARIFAS, salida: str = OUTPUT_CSV) -> pd.DataFrame:
    """
    Orquesta el pipeline ETL completo para TODOS los meses disponibles:
        1. Extracción  → obtiene todos los enlaces PDF (columna 'Periodo') de la web de CENS
        2. Iteración   → descarga y procesa cada PDF mes a mes con pdfplumber
        3. Carga       → consolida todo en un único CSV ordenado por Ciclo

    Returns:
        DataFrame consolidado: todos los meses × 4 niveles de tensión.
        Columnas: Fecha, Ciclo, Comercializador, Mercado, Nivel_Tension, Dueño_Red,
                  G, T, D, Cv, PR, R, CU
    """
    print("=" * 60)
    print("  PIPELINE ETL — TARIFAS CENS (TODOS LOS MESES)")
    print("=" * 60)

    # ── PASO 1: Obtener todos los enlaces PDF disponibles ────────
    try:
        enlaces = obtener_enlaces_pdf(url)
    except requests.exceptions.SSLError:
        print("  ⚠ SSL Error — reintentando sin verificación SSL (solo diagnóstico)...")
        import warnings
        warnings.warn("SSL verification disabled. Do not use in production.", UserWarning)
        respuesta = requests.get(url, headers=HEADERS, timeout=30, verify=False)
        soup = BeautifulSoup(respuesta.text, "html.parser")
        from urllib.parse import urljoin
        enlaces = []
        for tag_a in soup.find_all("a", href=True):
            href = tag_a["href"]
            if re.search(r"Tarifas.*\.pdf", href, re.I):
                href_limpio = href.split("?")[0]
                url_d = urljoin(url, href_limpio) if not href_limpio.startswith("http") \
                    else href_limpio
                nombre = url_d.split("/")[-1]
                enlaces.append((nombre, url_d))
        enlaces.sort(
            key=lambda t: re.search(r"\d{6}", t[0]).group()
            if re.search(r"\d{6}", t[0]) else t[0]
        )
    except Exception as e:
        print(f"\n✖ Error obteniendo enlaces: {e}")
        raise

    # ── PASO 2: Descargar y procesar cada PDF ───────────────────
    print(f"\n[ETL] Procesando {len(enlaces)} PDF(s)...\n")
    todos_los_df: list[pd.DataFrame] = []
    errores: list[str] = []

    for nombre_archivo, url_descarga in enlaces:
        ciclo_match = re.search(r"\d{6}", nombre_archivo)
        ciclo_label = ciclo_match.group() if ciclo_match else nombre_archivo
        print(f"── Ciclo {ciclo_label} ─────────────────────────────────")
        try:
            contenido = descargar_archivo(url_descarga, nombre_archivo)
            df_mes = procesar_pdf(contenido, nombre_archivo)
            todos_los_df.append(df_mes)
        except Exception as e:
            msg = f"Ciclo {ciclo_label}: {e}"
            print(f"  ✖ Error — {msg}")
            errores.append(msg)

    if not todos_los_df:
        raise RuntimeError(
            "No se procesó ningún archivo. "
            f"Errores registrados: {errores}"
        )

    # ── PASO 3: Consolidar y exportar ───────────────────────────
    df_final = pd.concat(todos_los_df, ignore_index=True)
    df_final = df_final.sort_values(
        by=["Ciclo", "Nivel_Tension"],
        key=lambda col: col.astype(str)
    ).reset_index(drop=True)

    generar_csv(df_final, salida)

    print(f"\n{'=' * 60}")
    print(f"  PIPELINE COMPLETADO")
    print(f"  Meses procesados : {len(todos_los_df)}/{len(enlaces)}")
    print(f"  Filas totales    : {len(df_final)}")
    if errores:
        print(f"  Errores          : {len(errores)} (ver arriba)")
    print("=" * 60)
    return df_final


# ─── PUNTO DE ENTRADA ────────────────────────────────────────────────────────
df_resultado = ejecutar_pipeline()


  PIPELINE ETL — TARIFAS CENS (TODOS LOS MESES)
[SCRAPING] Accediendo a: https://www.cens.com.co/clientes-y-usuarios/tarifas-de-energia
  ✔ Tabla encontrada con encabezados: ['Periodo', 'Rendimiento Financiero', 'Costo Opción Tarifaria']
  → PDF Tarifas: Tarifas_CENS_202601_.pdf  (ciclo 202601)
  → PDF Tarifas: Tarifas_CENS_202602_.pdf  (ciclo 202602)
  → PDF Tarifas: Tarifas_CENS_202603_.pdf  (ciclo 202603)
  → PDF Tarifas: Tarifas_CENS_202604_.pdf  (ciclo 202604)
  → PDF Tarifas: Tarifas_CENS_202605_.pdf  (ciclo 202605)

[SCRAPING] Total archivos PDF disponibles: 5

[ETL] Procesando 5 PDF(s)...

── Ciclo 202601 ─────────────────────────────────
  ↓ Descargando Tarifas_CENS_202601_.pdf ... 2539.0 KB
  Ciclo: 202601  →  Fecha: 2026-01-01
  Tabla CU detectada en página 4 (imagen → OCR)
  Inicializando OCR (primera vez: descarga modelos ~200 MB)...
  OCR listo.


C:\Users\geoff\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  OCR: 153 elemento(s) detectados
  Filas OCR agrupadas: 38
    Encabezados de nivel (fila 6): ['Componentes Cuv', '1-2,CENS', '1-2 Compartido', '1-2,Particular', 'Nivel 2', 'Nivel 3', 'Nivel 4']
    Niveles mapeados: {'1': 1, '2': 4, '3': 5, '4': 6}
    ⚠ Etiqueta no detectada → asignado como 'G' (valores: ['298.9047', '298.9047', '298.9047']…)
    ⚠ Etiqueta no detectada → asignado como 'T' (valores: ['52.9743', '52.9743', '52.9743']…)
  → DataFrame: 4 filas × 13 columnas
── Ciclo 202602 ─────────────────────────────────
  ↓ Descargando Tarifas_CENS_202602_.pdf ... 2529.6 KB
  Ciclo: 202602  →  Fecha: 2026-02-01
  Tabla CU detectada en página 4 (imagen → OCR)
  OCR: 149 elemento(s) detectados
  Filas OCR agrupadas: 38
    Encabezados de nivel (fila 6): ['Componentes Cuv', '1-2,CENS', '1-2 Compartido', '1-2,Particular', 'Nivel 2', 'Nivel 3', 'Nivel 4']
    Niveles mapeados: {'1': 1, '2': 4, '3': 5, '4': 6}
    ⚠ Etiqueta no detectada → asignado como 'G' (valores: ['271.7857', '271.785

: 

## 6. Verificación del Resultado

In [46]:
# Vista previa del DataFrame generado
print("Vista previa del CSV generado:")
print("-" * 80)
display(df_resultado)

Vista previa del CSV generado:
--------------------------------------------------------------------------------


,Fecha,Ciclo,Comercializador,Mercado,Nivel_Tension,Dueño_Red,G,T,D,Cv,PR,R,CU
0,2026-01-01,202601,CENS,CENS,1,100% OPERADOR,298.9047,52.9743,317.1201,132.3776,67.3480,19.3343,888.0590
1,2026-01-01,202601,CENS,CENS,2,100% OPERADOR,298.9047,52.9743,198.4778,120.1964,23.0861,19.3343,712.9736
2,2026-01-01,202601,CENS,CENS,3,100% OPERADOR,298.9047,52.9743,88.9777,121.7416,23.5973,19.3343,605.5299
3,2026-01-01,202601,CENS,CENS,4,100% OPERADOR,298.9047,52.9743,35.2509,85.2814,13.2552,19.3343,505.0008
4,2026-02-01,202602,CENS,CENS,1,100% OPERADOR,271.7857,50.6152,307.0274,149.8231,61.9511,21.6295,862.8320
5,2026-02-01,202602,CENS,CENS,2,100% OPERADOR,271.7857,50.6152,196.9665,123.8101,21.3698,21.6295,686.1768
6,2026-02-01,202602,CENS,CENS,3,100% OPERADOR,271.7857,50.6152,88.9525,98.2278,21.8640,21.6295,553.0747
7,2026-02-01,202602,CENS,CENS,4,100% OPERADOR,271.7857,50.6152,35.5022,89.4042,12.3567,21.6295,481.2935
8,2026-03-01,202603,CENS,CENS,1,100% OPERADOR,306.5637,NaN,300.5045,158.9335,71.1731,31.0329,925.8397
9,2026-03-01,202603,CENS,CENS,2,100% OPERADOR,306.5637,NaN,246.2446,124.1647,71.1731,31.0329,871.5798


In [ ]:
# Verificar que el CSV tiene el formato correcto
print("Primeras líneas del CSV exportado:")
print("-" * 80)
with open(OUTPUT_CSV, "r", encoding="utf-8-sig") as f:
    for i, linea in enumerate(f):
        print(linea.rstrip())
        if i >= 5:
            break

---
## 🔧 Modo Fallback — Cargar un PDF Local

Si el scraping no funciona (por cambios en la web, VPN corporativa, etc.), 
puedes cargar el archivo PDF manualmente y ejecutar solo la transformación.


In [ ]:
# ─── MODO FALLBACK: descomentar y ajustar la ruta del archivo PDF local ───────

# RUTA_PDF_LOCAL = "Tarifas_CENS_202601_.pdf"  # ← Cambiar por la ruta real

# with open(RUTA_PDF_LOCAL, "rb") as f:
#     contenido_local = f.read()

# nombre_local = os.path.basename(RUTA_PDF_LOCAL)
# df_fallback = procesar_pdf(contenido_local, nombre_local)
# generar_csv(df_fallback)
# display(df_fallback)

print("Modo fallback disponible — descomentar las líneas anteriores si es necesario.")
